In [20]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [45]:
import transformers
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments
from transformers import DataCollatorWithPadding, TextClassificationPipeline, Trainer

import re

In [22]:
ds = load_dataset("zeroshot/twitter-financial-news-sentiment")
ds

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 2388
    })
})

In [23]:
# printing first few samples
for i in range(10):
    print(f"Sample {i}: {ds['train'][i]['text']}")

Sample 0: $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT
Sample 1: $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.co/yGjpT2ReD3
Sample 2: $CX - Cemex cut at Credit Suisse, J.P. Morgan on weak building outlook https://t.co/KN1g4AWFIb
Sample 3: $ESS: BTIG Research cuts to Neutral https://t.co/MCyfTsXc2N
Sample 4: $FNKO - Funko slides after Piper Jaffray PT cut https://t.co/z37IJmCQzB
Sample 5: $FTI - TechnipFMC downgraded at Berenberg but called Top Pick at Deutsche Bank https://t.co/XKcPDilIuU
Sample 6: $GM - GM loses a bull https://t.co/tdUfG5HbXy
Sample 7: $GM: Deutsche Bank cuts to Hold https://t.co/7Fv1ZiFZBS
Sample 8: $GTT: Cowen cuts to Market Perform
Sample 9: $HNHAF $HNHPD $AAPL - Trendforce cuts iPhone estimate after Foxconn delay https://t.co/rlnEwzlzzS


In [24]:
# Filtering the text without Links
def filter_text(example):
    example['text'] = re.sub(r'https://\S+','', example['text'])
    return example

In [25]:
ds_filtered = ds.map(filter_text)
for i in range(10):
    print(f"Sample {i}: {ds_filtered['train'][i]['text']}")

Sample 0: $BYND - JPMorgan reels in expectations on Beyond Meat 
Sample 1: $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean 
Sample 2: $CX - Cemex cut at Credit Suisse, J.P. Morgan on weak building outlook 
Sample 3: $ESS: BTIG Research cuts to Neutral 
Sample 4: $FNKO - Funko slides after Piper Jaffray PT cut 
Sample 5: $FTI - TechnipFMC downgraded at Berenberg but called Top Pick at Deutsche Bank 
Sample 6: $GM - GM loses a bull 
Sample 7: $GM: Deutsche Bank cuts to Hold 
Sample 8: $GTT: Cowen cuts to Market Perform
Sample 9: $HNHAF $HNHPD $AAPL - Trendforce cuts iPhone estimate after Foxconn delay 


In [26]:
# Tokenizer
model_id = "meta-llama/Llama-3.2-1B"
tokenizer = AutoTokenizer.from_pretrained(model_id, model_max_length=256)

# setting the pad token
tokenizer.pad_token = tokenizer.eos_token

In [27]:
def tokenize(example, tokenizer):
    example = tokenizer(example['text'], padding=False, truncation=True)

    return example

In [28]:
# Checking the number of CPUs
from multiprocessing import cpu_count
cpu_count()

32

In [29]:
tokenized_ds = ds.map(tokenize, batched=True, num_proc=25, 
                               remove_columns=['text',],
                               fn_kwargs={"tokenizer": tokenizer}, )
print(tokenized_ds)

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 9543
    })
    validation: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 2388
    })
})


In [30]:
data_collator = DataCollatorWithPadding(tokenizer, padding=True)

In [31]:
model = AutoModelForSequenceClassification.from_pretrained(model_id,num_labels=3,
                                                           pad_token_id=tokenizer.eos_token_id,)

Some weights of LlamaForSequenceClassification were not initialized from the model checkpoint at meta-llama/Llama-3.2-1B and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [32]:
print(model)

LlamaForSequenceClassification(
  (model): LlamaModel(
    (embed_tokens): Embedding(128256, 2048, padding_idx=128001)
    (layers): ModuleList(
      (0-15): 16 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=512, bias=False)
          (v_proj): Linear(in_features=2048, out_features=512, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (up_proj): Linear(in_features=2048, out_features=8192, bias=False)
          (down_proj): Linear(in_features=8192, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((20

In [33]:
model.config.id2label = {0:"Bearish", 1:"Bullish", 2:"Neutral"}

In [36]:
from peft import LoraConfig, TaskType

lora_config = LoraConfig(
    task_type= TaskType.SEQ_CLS,
    lora_alpha=32,
    lora_dropout= 0.05,
    r= 16, # lora attention rank
    target_modules=["q_proj", "v_proj"],
    inference_mode= False
)

In [38]:
from peft import get_peft_model

lora_model = get_peft_model(model, lora_config)
lora_model.print_trainable_parameters()


trainable params: 1,710,080 || all params: 1,237,530,624 || trainable%: 0.1382


In [39]:
import numpy as np
import evaluate

metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

In [42]:
training_args = TrainingArguments( output_dir='llama32_imdb_ft_lora',
                                  eval_strategy="steps",
                                  eval_steps=100,
                                  num_train_epochs=1,
                                  per_device_train_batch_size=4,
                                  per_device_eval_batch_size=4,
                                  bf16=False,
                                  fp16=True,
                                  tf32=False,
                                  gradient_accumulation_steps=1,
                                  adam_beta1=0.9,
                                  adam_beta2=0.999,
                                  learning_rate=2e-5,
                                  weight_decay=0.01,
                                  logging_dir='logs',
                                  logging_strategy="steps",
                                  logging_steps = 100,
                                  save_steps=500,
                                  save_total_limit=20,
                                  report_to='none',
                                )

In [48]:
trainer = Trainer(model=lora_model,
                  args = training_args,
                 train_dataset=tokenized_ds["train"],
                 eval_dataset=tokenized_ds["validation"],
                 compute_metrics=compute_metrics,
                 data_collator = data_collator)

No label_names provided for model class `PeftModelForSequenceClassification`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.


In [49]:
result = trainer.train()

Step,Training Loss,Validation Loss,Accuracy
100,1.116900,1.058641,0.624372
200,0.949500,0.983205,0.642379
300,0.984200,0.946661,0.653685
400,0.945300,0.855136,0.681323
500,0.837200,0.805897,0.680905
600,0.826400,0.757684,0.705611
700,0.750000,0.734902,0.715662
800,0.727200,0.681698,0.731993
900,0.657000,0.704972,0.742881
1000,0.716700,0.613730,0.769682


In [50]:
classifier = TextClassificationPipeline(model=lora_model,
                                        tokenizer = tokenizer,
                                        framework='pt',
                                        task="sentiment-analysis",
                                        device = "cuda"
                                       )

Device set to use cuda


Sentences which have no correlation to the market

In [51]:
text = "The movie is good."
prediction = classifier(text)
print(prediction)

[{'label': 'Neutral', 'score': 0.9978450536727905}]


In [52]:
text = "The movie is really bad..nothing new to hook us"
prediction = classifier(text)
print(prediction)

[{'label': 'Neutral', 'score': 0.9830896854400635}]


In [53]:
text = "Very bad movie with no good story"
prediction = classifier(text)
print(prediction)

[{'label': 'Neutral', 'score': 0.9859936237335205}]


Sentences which could have an effect on Market

In [58]:
text = " $FTI - TechnipFMC downgraded at Berenberg but called Top Pick at Deutsche Bank "
prediction = classifier(text)
print(prediction)

[{'label': 'Bearish', 'score': 0.7777025103569031}]
